# Reaction path with AFIR + MACE through NVIDIA ALCHEMI, on a platform GPU

Runs the Claisen-rearrangement search of an input molecule as a platform job: reactant relaxation, an
artificial-force (AFIR) push along the forming bond, a dimer refinement of the transition state, and a check
for the imaginary mode. Energies and forces come from MACE evaluated through the
[NVIDIA ALCHEMI toolkit](https://github.com/NVIDIA/nvalchemi-toolkit).

The workflow itself lives on the platform and is looked up by name, so it stays editable in the Workflow
Designer. This notebook selects the material, submits the job, shows the published results, and saves the
transition state back as a material.

The same science runs on a laptop in `local/reaction_path_afir_alchemi.ipynb`.

## 1. Set up the environment and parameters

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime

from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None  # set to use an organization as the owner, otherwise the personal account

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "allyl vinyl ether"  # loaded from FOLDER, otherwise from Materials Standata

# 4. Workflow parameters — the workflow is looked up on the platform, not built here
WORKFLOW_NAME = "AFIR ALCHEMI - Material IO (WIP)"

# Files the workflow's execution unit declares as file_content results
RESULT_FILES = {
    "results.csv": "csv",
    "afir_energy_profile.png": "image",
    "afir_bond_distances.png": "image",
    "afir_path.csv": "csv",
    "structures.json": "text",
    "transition_state.json": "text",
    "transition_state.xyz": "text",
}

# 5. Compute parameters
CLUSTER_NAME = "cluster-003"
QUEUE_NAME = QueueName.GSF  # H100 nodes on cluster-003 (GOF on-demand, GSF spot)
PPN = 40

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

## 2. Authenticate and initialize API client

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate


await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create material

The workflow's IO unit fetches this material into the job, and the script builds its ASE molecule from it.

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.standata.materials import Materials

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)
print(f"Material ID: {saved_material.id}  ({saved_material.name})")

## 4. Find the workflow on the platform

Editing the workflow in the Workflow Designer changes what this notebook submits — no code change here.

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from mat3ra.wode.workflows import Workflow

matches = client.workflows.list({"name": WORKFLOW_NAME, "owner._id": ACCOUNT_ID})
assert matches, (
    f"No workflow named {WORKFLOW_NAME!r} under this account. "
    "Check the name in the Workflows page, or that the account is the owner."
)
if len(matches) > 1:
    print(f"⚠️  {len(matches)} workflows share this name; using the most recently updated")
    matches = sorted(matches, key=lambda w: w.get("updatedAt", ""), reverse=True)

workflow_document = matches[0]  # create_job needs the plain dict, not the wode object
saved_workflow = Workflow.create(workflow_document)
print(f"Workflow ID: {saved_workflow.id}  ({saved_workflow.name})")

visualize_workflow(saved_workflow)

### 4.2. Check what the workflow says it publishes

A file the script writes but the unit does not declare is never uploaded, and a declared name that does not
match a written file is silently dropped — so it is worth comparing the two before spending a job.

In [ ]:
declared = {
    r["basename"]: r.get("filetype")
    for sw in saved_workflow.to_dict()["subworkflows"]
    for u in sw["units"]
    for r in (u.get("results") or [])
    if r.get("name") == "file_content" and "basename" in r
}
print(f"declared by the workflow: {len(declared)}")
for basename, filetype in RESULT_FILES.items():
    mark = "ok" if declared.get(basename) == filetype else "MISSING or wrong filetype"
    print(f"  {basename:28} {mark}")
extra = set(declared) - set(RESULT_FILES)
if extra:
    print(f"  declared but not expected: {sorted(extra)}")

## 5. Create the compute configuration

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration for the job

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job with material and workflow configuration

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.ui import display_JSON
import copy

from mat3ra.utils.namespace import dict_to_namespace_recursive

job_name = f"{WORKFLOW_NAME} {saved_material.name} {timestamp}"
job_response = create_job(
    api_client=client,
    materials=[saved_material_response],  # dicts, not entity objects: create_job subscripts them
    workflow=copy.deepcopy(workflow_document),  # create_job mutates it (pops _id)
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace_recursive(job_response)
job_id = job._id
print(f"✅ Job created: {job_id}")

# create_job embeds a copy of the workflow; if the wrong one is attached the job runs silently
# against the account default, which looks like a working job with unrelated results.
attached = job_response["workflow"]["name"]
assert attached == WORKFLOW_NAME, f"job got workflow {attached!r}, expected {WORKFLOW_NAME!r}"
print(f"✅ Workflow attached: {attached}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Results

An empty list with a finished job means the files were never uploaded — a cluster problem rather than a
workflow one. The *Files* tab of the job distinguishes the two: if `script.py` is missing there too, nothing
about the workflow is being tested.

In [ ]:
files = {f["key"].rsplit("/", 1)[-1]: f["signedUrl"] for f in client.jobs.list_files(job_id)}
print(f"files uploaded: {len(files)}")
for basename in RESULT_FILES:
    print(f"  {basename:28} {'published' if basename in files else 'MISSING'}")

### 8.2. Which device did the work

A fast job is not evidence of a GPU. These lines are.

In [ ]:
from mat3ra.notebooks_utils.io import read_from_url

stdout_key = next((k for k in files if k.endswith(".out")), None)
if stdout_key:
    for line in (await read_from_url(files[stdout_key])).splitlines():
        if line.startswith(("PyTorch version", "CUDA compiled", "Is CUDA available",
                            "Running simulation on:", "Reactant Energy",
                            "Refined Activation Energy", "Found ")):
            print(line)

### 8.3. The figures and the numbers

In [ ]:
from IPython.display import Image, display

for basename in ("afir_energy_profile.png", "afir_bond_distances.png"):
    if basename in files:
        display(Image(data=await read_from_url(files[basename], as_bytes=True)))

if "results.csv" in files:
    print(await read_from_url(files["results.csv"]))

## 9. Save the reaction path as materials

The job writes `structures.json` — the input, every AFIR image and the refined transition state, each a
full Mat3ra material with lattice, basis, metadata and hashes. A python execution unit cannot register a
material itself, so that happens here; the result is a set of real materials, each usable as input to a
next job.


In [ ]:
import json

assert "structures.json" in files, "the job did not publish structures.json"

path_structures = json.loads(await read_from_url(files["structures.json"]))
print(f"{len(path_structures)} structures along the path\n")

saved_structures = []
for entry in path_structures:
    mat = Material.create(entry["material"])
    response = get_or_create_material(client, mat, ACCOUNT_ID)
    saved = Material.create(response)
    saved_structures.append(saved)
    energy = entry["energy_eV"]
    energy_text = f"{energy:.3f} eV" if energy is not None else "-"
    print(f"  {entry['label']:38} {energy_text:>12}   {saved.id}")

print(f"\n✅ {len(saved_structures)} materials saved under account {ACCOUNT_ID}")
